In [ ]:
import pandas as pd
import os
from dataframe_modify.add_positions import add_positions
from dataframe_modify.swap_perspective import swap_y
from dataframe_modify.swap_perspective import generalize_perspective_sb360
from dataframe_modify.grid_areas import add_grid_areas
from sequences.get_sequences import get_sequences, get_corners_sequences, get_sequences_by_possession
from sequences.find_fast_actions import find_fast_actions_forward_perspective
from fast_actions.get_dataframe import get_fast_action_row
from corners.metric import evaluate_corner
from corners.get_dataframe import get_df_row

In [ ]:
def prepare_matches_from_pkl(pkl_path):
    pkl_preprocessed_path = pkl_path[:-4] + '_preprocessed.pkl'
    if not os.path.exists(pkl_preprocessed_path):
        matches = pd.read_pickle(pkl_path)
        matches = add_positions(matches)
        matches, matches_without_sb360 = swap_y(matches)
        for match_id in matches_without_sb360:
            matches = matches[matches['match_id'] != match_id]
        matches = generalize_perspective_sb360(matches)
        matches = add_grid_areas(matches)
        return matches
    return pd.read_pickle(pkl_preprocessed_path)

In [ ]:
params = {
    'pass_to_zone_I': -0.8,
    'pass_to_zone_II': 0.1,
    'pass_to_zone_III': 0.2,
    'pass_incomplete': -0.5,
    'pass_shot_assist': 0.4,
    'pass_goal_assist': 0.4,
    'pass_cross': 0.2,
    'pass_by_opponent': -0.5,
    'shot_goal': 2,
    'shot_saved': 1,
    'shot_blocked': 0.5,
    'shot_off_target': 0.5,
    'shot_post': 1,
    'shot_saved_to_post': 1,
    'shot_saved_off_target': 0.5,
    'clearance_by_opponent': -0.3,
    'ball_recovery_by_opponent': -0.3,
    'ball_recovery_by_own_team': 0.5,
    'foul_committed_by_own_team': -0.5,
    'foul_committed_by_own_team_with_card': -1,
    'foul_committed_by_opponent': 0.5,
    'foul_committed_by_opponent_with_penalty': 1,
}

In [ ]:
esa_21_22_matches = prepare_matches_from_pkl('data/esa_21_22_matches.pkl')

In [ ]:
sequences_21_22 = get_sequences_by_possession(esa_21_22_matches)
corners_21_22 = get_corners_sequences(sequences_21_22)

In [ ]:
corners_21_22 =  [evaluate_corner(corner, params) for corner in corners_21_22]

In [ ]:
def df_print_corner(df):
    corner_df = df[['period', 'timestamp', 'type', 'team', 'player', 'event', 'corner_score']]
    return corner_df

In [ ]:
df_print_corner(corners_21_22[-3])#.to_csv('corner_2.csv', index=False)

In [ ]:
corners_21_22[-1].columns

In [ ]:
def get_corner_obv(corner):
    obv = 0
    # sum obv_total_net for all events in corner
    for i in range(len(corner)):
        if corner.iloc[i]['obv_for_net']:
            obv += float(corner.iloc[i]['obv_for_net'])
    return obv

In [ ]:
def get_corner_xg(corner):
    xg = 0
    for i in range(len(corner)):
        if corner.iloc[i]['type'] == 'Shot':
            if corner.iloc[i]['event'].get('statsbomb_xg') != None:
                print(corner.iloc[i]['event'].get('statsbomb_xg'))
                xg += float(corner.iloc[i]['event'].get('statsbomb_xg'))
    return xg

In [ ]:
len(corners_21_22)

In [ ]:
import xgboost as xgb

In [ ]:
from corners.get_dataframe import get_df_row
xgb_df = pd.DataFrame()
xgb_df.append([get_df_row(corner) for corner in corners_21_22])